In [1]:
import torch

def explore_checkpoint(checkpoint):
    for key in checkpoint.keys():
        print(f"\nKey: {key}")
        value = checkpoint[key]
        
        if isinstance(value, dict):
            print(f"Type: Dictionary with {len(value)} keys")
            for sub_key, sub_value in list(value.items())[:3]:  # Limit to 3 keys
                print(f"  - {sub_key}: {type(sub_value)}")
                if isinstance(sub_value, torch.Tensor):
                    print(f"    Shape: {sub_value.shape}")
        
        elif isinstance(value, list):
            print(f"Type: List with {len(value)} items")
            if value:
                print(f"  First Item Type: {type(value[0])}")
        
        elif isinstance(value, torch.Tensor):
            print(f"Type: Tensor with shape {value.shape}")
        
        else:
            print(f"Type: {type(value)}")
            print(f"  Value: {value}")

# Load the checkpoint
checkpoint = torch.load("/data/geraugi/plural/checkpoints/ofa_base.pt", map_location="cpu")
plural_ckpt = torch.load("/data/geraugi/plural/checkpoints/PLURAL.pt", map_location="cpu")
explore_checkpoint(plural_ckpt)
# Inspect the keys
#print(checkpoint.keys())

/data/geraugi/anaconda3/envs/plural/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Key: args
Type: <class 'NoneType'>
  Value: None

Key: cfg
Type: Dictionary with 20 keys
  - _name: <class 'NoneType'>
  - common: <class 'dict'>
  - common_eval: <class 'dict'>

Key: model
Type: Dictionary with 967 keys
  - encoder.version: <class 'torch.Tensor'>
    Shape: torch.Size([1])
  - encoder.token_rp_bucket: <class 'torch.Tensor'>
    Shape: torch.Size([1024, 1024])
  - encoder.image_rp_bucket: <class 'torch.Tensor'>
    Shape: torch.Size([1765, 1765])

Key: criterion
Type: <class 'NoneType'>
  Value: None

Key: optimizer_history
Type: List with 10 items
  First Item Type: <class 'dict'>

Key: task_state
Type: Dictionary with 0 keys

Key: extra_state
Type: Dictionary with 5 keys
  - metrics: <class 'collections.OrderedDict'>
  - previous_training_time: <class 'torch.Tensor'>
    Shape: torch.Size([])
  - train_iterator: <class 'dict'>

Key: last_optimizer_state
Type: Dictionary with 3 keys
  - state: <class 'dict'>
  - param_groups: <class 'list'>
  - loss_scale: <class 'fl

In [18]:
def compare_cfg_structures(cfg1, cfg2, cfg1_name="PLURAL", cfg2_name="OFA", path="root"):
    """
    Compare the structures of two nested configuration dictionaries.
    
    Args:
        cfg1, cfg2 (dict): Configuration dictionaries to compare.
        cfg1_name, cfg2_name (str): Names of the configs (for labeling in output).
        path (str): Current key path (used for nested comparisons).
    """
    cfg1_keys = set(cfg1.keys())
    cfg2_keys = set(cfg2.keys())
    
    # Keys present in cfg1 but missing in cfg2
    for key in cfg1_keys - cfg2_keys:
        print(f"[{path}] Key '{key}' found in {cfg1_name}, but not in {cfg2_name}")
    
    # Keys present in cfg2 but missing in cfg1
    for key in cfg2_keys - cfg1_keys:
        print(f"[{path}] Key '{key}' found in {cfg2_name}, but not in {cfg1_name}")
    
    # Common keys
    for key in cfg1_keys & cfg2_keys:
        value1 = cfg1[key]
        value2 = cfg2[key]
        current_path = f"{path} -> {key}"
        
        # Check type
        if type(value1) != type(value2):
            print(f"[{current_path}] Type mismatch: {cfg1_name}={type(value1)}, {cfg2_name}={type(value2)}")
        
        # If both are dictionaries, recurse
        elif isinstance(value1, dict) and isinstance(value2, dict):
            compare_cfg_structures(value1, value2, cfg1_name, cfg2_name, current_path)
        
        # If both are lists, compare lengths and types
        elif isinstance(value1, list) and isinstance(value2, list):
            if len(value1) != len(value2):
                print(f"[{current_path}] List length mismatch: {cfg1_name}={len(value1)}, {cfg2_name}={len(value2)}")
            else:
                for i, (v1, v2) in enumerate(zip(value1, value2)):
                    if type(v1) != type(v2):
                        print(f"[{current_path}[{i}]] Type mismatch in list: {cfg1_name}={type(v1)}, {cfg2_name}={type(v2)}")
        
        # Otherwise, compare values directly
        else:
            try:
                if value1 != value2:
                    print(f"[{current_path}] Value mismatch: {cfg1_name}={value1}, {cfg2_name}={value2}")
            except Exception as e:
                print(f"Problems comparing values in path {current_path}: {e}")

# Example usage
# Assume plural_cfg and ofa_cfg are loaded dictionaries for PLURAL and OFA configurations.
compare_cfg_structures(plural_ckpt, checkpoint)

[root -> extra_state] Key 'best' found in PLURAL, but not in OFA
[root -> extra_state -> previous_training_time] Value mismatch: PLURAL=1947266.775811337, OFA=1854561.424985312
[root -> extra_state -> metrics] Key 'valid' found in PLURAL, but not in OFA
[root -> extra_state -> metrics -> default] List length mismatch: PLURAL=21, OFA=22
[root -> extra_state -> metrics -> train_inner] List length mismatch: PLURAL=20, OFA=21
[root -> extra_state -> metrics -> train] List length mismatch: PLURAL=20, OFA=21
[root -> extra_state -> val_loss] Type mismatch: PLURAL=<class 'float'>, OFA=<class 'NoneType'>
[root -> extra_state -> train_iterator -> iterations_in_epoch] Value mismatch: PLURAL=0, OFA=16072
[root -> extra_state -> train_iterator -> epoch] Value mismatch: PLURAL=2, OFA=48
[root -> extra_state -> train_iterator -> shuffle] Value mismatch: PLURAL=True, OFA=False
[root -> cfg] Key 'simul_type' found in OFA, but not in PLURAL
[root -> cfg -> common -> fp16_scale_window] Value mismatch: P

In [15]:
for key in plural_ckpt['cfg'].keys():
    print(f'#############\n {key} \n#############')
    if plural_ckpt['cfg'][key] != None:
        if key != 'model':
            explore_checkpoint(plural_ckpt['cfg'][key])

#############
 _name 
#############
#############
 common 
#############

Key: _name
Type: <class 'NoneType'>
  Value: None

Key: no_progress_bar
Type: <class 'bool'>
  Value: False

Key: log_interval
Type: <class 'int'>
  Value: 10

Key: log_format
Type: <class 'str'>
  Value: simple

Key: log_file
Type: <class 'NoneType'>
  Value: None

Key: tensorboard_logdir
Type: <class 'NoneType'>
  Value: None

Key: wandb_project
Type: <class 'NoneType'>
  Value: None

Key: azureml_logging
Type: <class 'bool'>
  Value: False

Key: seed
Type: <class 'int'>
  Value: 1

Key: cpu
Type: <class 'bool'>
  Value: False

Key: tpu
Type: <class 'bool'>
  Value: False

Key: bf16
Type: <class 'bool'>
  Value: False

Key: memory_efficient_bf16
Type: <class 'bool'>
  Value: False

Key: fp16
Type: <class 'bool'>
  Value: True

Key: memory_efficient_fp16
Type: <class 'bool'>
  Value: False

Key: fp16_no_flatten_grads
Type: <class 'bool'>
  Value: False

Key: fp16_init_scale
Type: <class 'int'>
  Value: 128

Key:

In [12]:
explore_checkpoint(checkpoint['model'])


Key: encoder.version
Type: Tensor with shape torch.Size([1])

Key: encoder.token_rp_bucket
Type: Tensor with shape torch.Size([1024, 1024])

Key: encoder.image_rp_bucket
Type: Tensor with shape torch.Size([1765, 1765])

Key: encoder.embed_tokens.weight
Type: Tensor with shape torch.Size([59457, 768])

Key: encoder.layernorm_embedding.weight
Type: Tensor with shape torch.Size([768])

Key: encoder.layernorm_embedding.bias
Type: Tensor with shape torch.Size([768])

Key: encoder.type_embedding.weight
Type: Tensor with shape torch.Size([2, 768])

Key: encoder.embed_images.conv1.weight
Type: Tensor with shape torch.Size([64, 3, 7, 7])

Key: encoder.embed_images.bn1.weight
Type: Tensor with shape torch.Size([64])

Key: encoder.embed_images.bn1.bias
Type: Tensor with shape torch.Size([64])

Key: encoder.embed_images.bn1.running_mean
Type: Tensor with shape torch.Size([64])

Key: encoder.embed_images.bn1.running_var
Type: Tensor with shape torch.Size([64])

Key: encoder.embed_images.bn1.num_ba

In [3]:
print(plural_ckpt['last_optimizer_state'].keys())
explore_checkpoint(plural_ckpt['last_optimizer_state'])
print(plural_ckpt['last_optimizer_state']['state'])
print(plural_ckpt['last_optimizer_state']['param_groups'])
print(plural_ckpt['last_optimizer_state']['loss_scale'])

dict_keys(['state', 'param_groups', 'loss_scale'])

Key: state
Type: Dictionary with 1 keys
  - 0: <class 'dict'>

Key: param_groups
Type: List with 1 items
  First Item Type: <class 'dict'>

Key: loss_scale
Type: <class 'float'>
  Value: 64.0
{0: {'step': 672, 'exp_avg': tensor([4.2781e-06, 1.3723e-05, 4.9378e-06,  ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00]), 'exp_avg_sq': tensor([1.4835e-10, 1.3191e-09, 1.0310e-10,  ..., 0.0000e+00, 0.0000e+00,
        0.0000e+00])}}
[{'lr': 1.661721068249258e-05, 'betas': (0.9, 0.999), 'eps': 1e-08, 'weight_decay': 0.01, 'amsgrad': False, 'params': [0]}]
64.0


In [4]:
checkpoint=plural_ckpt
print(checkpoint['extra_state'].keys())
explore_checkpoint(checkpoint['extra_state'])
print("### TRAIN ITERATOR ###")
explore_checkpoint(checkpoint['extra_state']['train_iterator'])
print("### METRICS ###")
explore_checkpoint(checkpoint['extra_state']['metrics'])
for key in checkpoint['extra_state']['metrics']:
    print(f'\n### {key} ###\n')
    for element in checkpoint['extra_state']['metrics'][key]:
        print(element)
print(checkpoint['extra_state']['previous_training_time'])

dict_keys(['metrics', 'previous_training_time', 'train_iterator', 'val_loss', 'best'])

Key: metrics
Type: Dictionary with 4 keys
  - default: <class 'list'>
  - train: <class 'list'>
  - train_inner: <class 'list'>

Key: previous_training_time
Type: Tensor with shape torch.Size([])

Key: train_iterator
Type: Dictionary with 4 keys
  - version: <class 'int'>
  - epoch: <class 'int'>
  - iterations_in_epoch: <class 'int'>

Key: val_loss
Type: <class 'float'>
  Value: 1.731

Key: best
Type: <class 'float'>
  Value: 1.731
### TRAIN ITERATOR ###

Key: version
Type: <class 'int'>
  Value: 2

Key: epoch
Type: <class 'int'>
  Value: 2

Key: iterations_in_epoch
Type: <class 'int'>
  Value: 0

Key: shuffle
Type: <class 'bool'>
  Value: True
### METRICS ###

Key: default
Type: List with 21 items
  First Item Type: <class 'tuple'>

Key: train
Type: List with 20 items
  First Item Type: <class 'tuple'>

Key: train_inner
Type: List with 20 items
  First Item Type: <class 'tuple'>

Key: valid
Type: 

In [8]:
for i in range(10):
    print(checkpoint['optimizer_history'][i])
#explore_checkpoint(checkpoint['optimizer_history'])

{'criterion_name': 'CrossEntropyCriterion', 'optimizer_name': 'FP16Optimizer', 'lr_scheduler_state': {'best': 0.897}, 'num_updates': 240000}
{'criterion_name': 'CrossEntropyCriterion', 'optimizer_name': 'FP16Optimizer', 'lr_scheduler_state': {'best': 0.862}, 'num_updates': 489993}
{'criterion_name': 'CrossEntropyCriterion', 'optimizer_name': 'FP16Optimizer', 'lr_scheduler_state': {'best': 0.862}, 'num_updates': 500000}
{'criterion_name': 'AjustLabelSmoothedCrossEntropyCriterion', 'optimizer_name': 'FP16Optimizer', 'lr_scheduler_state': {'best': None}, 'num_updates': 60000}
{'criterion_name': 'AjustLabelSmoothedCrossEntropyCriterion', 'optimizer_name': 'FP16Optimizer', 'lr_scheduler_state': {'best': None}, 'num_updates': 78000}
{'criterion_name': 'AjustLabelSmoothedCrossEntropyCriterion', 'optimizer_name': 'FP16Optimizer', 'lr_scheduler_state': {'best': None}, 'num_updates': 162000}
{'criterion_name': 'AjustLabelSmoothedCrossEntropyCriterion', 'optimizer_name': 'FP16Optimizer', 'lr_sche

In [5]:
print(checkpoint['cfg'].keys())
explore_checkpoint(checkpoint['cfg'])

dict_keys(['_name', 'common', 'common_eval', 'distributed_training', 'dataset', 'optimization', 'checkpoint', 'bmuf', 'generation', 'eval_lm', 'interactive', 'model', 'task', 'criterion', 'optimizer', 'lr_scheduler', 'scoring', 'bpe', 'tokenizer', 'ema', 'simul_type'])

Key: _name
Type: <class 'NoneType'>
  Value: None

Key: common
Type: Dictionary with 36 keys
  - _name: <class 'NoneType'>
  - no_progress_bar: <class 'bool'>
  - log_interval: <class 'int'>

Key: common_eval
Type: Dictionary with 6 keys
  - _name: <class 'NoneType'>
  - path: <class 'NoneType'>
  - post_process: <class 'NoneType'>

Key: distributed_training
Type: Dictionary with 39 keys
  - _name: <class 'NoneType'>
  - distributed_world_size: <class 'int'>
  - distributed_num_procs: <class 'int'>

Key: dataset
Type: Dictionary with 25 keys
  - _name: <class 'NoneType'>
  - num_workers: <class 'int'>
  - skip_invalid_size_inputs_valid_test: <class 'bool'>

Key: optimization
Type: Dictionary with 10 keys
  - _name: <cla

In [11]:
for key in checkpoint['cfg'].keys():
    print(f'#############\n {key} \n#############')
    if checkpoint['cfg'][key] != None:
        if key != 'model':
            explore_checkpoint(checkpoint['cfg'][key])

#############
 _name 
#############
#############
 common 
#############

Key: _name
Type: <class 'NoneType'>
  Value: None

Key: no_progress_bar
Type: <class 'bool'>
  Value: False

Key: log_interval
Type: <class 'int'>
  Value: 10

Key: log_format
Type: <class 'str'>
  Value: simple

Key: log_file
Type: <class 'NoneType'>
  Value: None

Key: tensorboard_logdir
Type: <class 'NoneType'>
  Value: None

Key: wandb_project
Type: <class 'NoneType'>
  Value: None

Key: azureml_logging
Type: <class 'bool'>
  Value: False

Key: seed
Type: <class 'int'>
  Value: 1

Key: cpu
Type: <class 'bool'>
  Value: False

Key: tpu
Type: <class 'bool'>
  Value: False

Key: bf16
Type: <class 'bool'>
  Value: False

Key: memory_efficient_bf16
Type: <class 'bool'>
  Value: False

Key: fp16
Type: <class 'bool'>
  Value: True

Key: memory_efficient_fp16
Type: <class 'bool'>
  Value: False

Key: fp16_no_flatten_grads
Type: <class 'bool'>
  Value: False

Key: fp16_init_scale
Type: <class 'int'>
  Value: 128

Key:

In [15]:
print(checkpoint['cfg']['optimization']['stop_min_lr'])
print(checkpoint['cfg']['optimizer']['lr'])
print(checkpoint['cfg']['lr_scheduler']['lr'])
print(checkpoint['cfg']['optimization']['update_freq'])
print(checkpoint['cfg']['distributed_training']['distributed_world_size'])
print(checkpoint['cfg']['dataset']['batch_size'])

-1.0
[0.0001]
[0.0001]
[4]
3
16
